# Шаг 07 — память после перезапуска

- `compile(checkpointer=SqliteSaver(...))` — граф сохраняет состояние в SQLite после каждого узла;
- `thread_id` в конфиге — имя разговора. Историю мы больше не храним сами: отправляем только **новое** сообщение, остальное граф поднимет из базы.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENROUTER_API_KEY and MODEL from .env

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",   # OpenRouter speaks the OpenAI protocol
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
from tools import list_files, read_file

TOOLS = [list_files, read_file]

SYSTEM = ("Ты ассистент по коду. Смотри реальные файлы через инструменты, "
          "ничего не выдумывай. Отвечай кратко.")

from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

llm_with_tools = llm.bind_tools(TOOLS)


def agent(state: MessagesState) -> dict:
    """One model step: read the whole history, return the new message."""
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(TOOLS))              # runs the requested tools
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # → "tools" or END
builder.add_edge("tools", "agent")

print("builder ready — we compile it below, with a checkpointer")

Подключаем SQLite. Удаляем старую базу, чтобы демо начиналось с чистого листа.

In [ ]:
import sqlite3
from pathlib import Path

from langgraph.checkpoint.sqlite import SqliteSaver

DB = Path("memory.sqlite")
DB.unlink(missing_ok=True)                      # fresh demo

graph = builder.compile(checkpointer=SqliteSaver(sqlite3.connect(DB, check_same_thread=False)))

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

def print_update(update: dict) -> None:
    """Show what each graph node did."""
    for node, data in update.items():
        for msg in data["messages"]:
            if node == "agent" and msg.tool_calls:
                for call in msg.tool_calls:
                    print(f"  [agent] → {call['name']}({call['args']})")
            elif node == "tools":
                print(f"  [tools] {msg.name} done")
            else:
                print("ai ›", msg.content)


def ask(graph, text: str, thread: str = "main") -> None:
    config = {"configurable": {"thread_id": thread}}
    new = [HumanMessage(text)]
    if not graph.get_state(config).values:       # brand-new thread
        new.insert(0, SystemMessage(SYSTEM))
    for update in graph.stream({"messages": new}, config, stream_mode="updates"):
        print_update(update)

In [ ]:
ask(graph, "Меня зовут Женя, мой проект — парсер логов. Просто ответь ок.")

## «Перезапуск»

Создаём **новый** граф с новым подключением к тому же файлу — как будто процесс перезапустили. (Можно и правда сделать Restart Kernel и выполнить ячейки заново без строки `unlink`.)

In [ ]:
graph_after_restart = builder.compile(
    checkpointer=SqliteSaver(sqlite3.connect(DB, check_same_thread=False))
)
ask(graph_after_restart, "Как меня зовут и какой у меня проект? Одной строкой.")

Другой `thread_id` — другой разговор, там пусто:

In [ ]:
ask(graph_after_restart, "Как меня зовут? Если не знаешь — так и скажи.", thread="other")

Что лежит в базе: последнее состояние нити и все сохранённые шаги (чекпоинты).

In [ ]:
config = {"configurable": {"thread_id": "main"}}
state = graph_after_restart.get_state(config)
print("messages in thread:", len(state.values["messages"]))
print("checkpoints saved: ", len(list(graph_after_restart.get_state_history(config))))